# Gemma 4: Measure Before You Tune

**Abstract.** A developer agent needs a trustworthy grader, an inference path that matches training and evidence that a lower loss improves repairs. Paired controls leave **114 of 129 public tasks** sound in our frozen local environment. A served-prompt reconstruction matches **936 of 936 prompt lengths**, exposing a historical mismatch that discarded past reasoning. A trained low-rank adaptation (LoRA) debug run lowers Rich action loss by **40.7%**, yet resolved and unresolved traces improve almost equally and task resolution falls from **4/32 to 2/32**. The adapter receives a **NO-GO**. Together with a rollout error audit, these results show why rendering checks, negative controls and task-level regression gates should precede longer training.

Measurements use public tasks and our frozen local stack; Rich is the sole held-out repository. The rendering result concerns the inspected historical wheels: the organizer subsequently reported a reasoning-persistence fix. The archive built at the end reproduces the public **0.12** coder/analyzer bundle by Roman Rozen as the reference agent for these measurements; this notebook's own Public Leaderboard (LB) score is shown on the notebook page. [Official evaluation](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview/evaluation) · [Task description and harness](https://www.kaggle.com/competitions/gemma-4-developer-agent/data).


In [ ]:
from pathlib import Path
import hashlib, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, Markdown

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
def dataset_file(name):
    matches = sorted(Path('/kaggle/input').rglob(name))
    assert len(matches) == 1, f'Expected one attached {name}: {matches}'
    return matches[0]
A = dataset_file('task_soundness.csv').parent
K = dataset_file('gate_measurements.csv').parent
for folder in [A, K]:
    manifest = json.loads((folder/'manifest.json').read_text())
    for f in manifest['files']:
        assert hashlib.sha256((folder/f['name']).read_bytes()).hexdigest() == f['sha256'], f['name']
audit = pd.read_csv(A/'task_soundness.csv')
runs = pd.read_csv(A/'local_agent_results.csv')
summary = pd.read_csv(A/'series_summary.csv')
runtime = pd.read_csv(A/'runtime_scenarios.csv')
gate = pd.read_csv(K/'gate_measurements.csv').set_index('measurement')['value']
protocol = json.loads((A/'protocol.json').read_text())
assert len(audit)==129 and audit.task_id.is_unique and audit.sound.sum()==114
assert len(runs)==177 and not runs.duplicated(['series','task_id']).any()
assert set(runs.task_id) == set(audit.task_id)
assert not set(audit.query("split == 'development'").repo) & set(audit.query("split == 'held_out'").repo)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold','axes.titlelocation':'left','axes.labelcolor':'#253b4a','text.color':'#253b4a','axes.edgecolor':'#bbc6ca','figure.facecolor':'#ffffff','axes.facecolor':'#ffffff','savefig.facecolor':'#ffffff'})
TEAL, BLUE, AMBER, RED, GRAY = '#23877c','#447da9','#c18d43','#bd625c','#d5dee1'
def save(fig, name):
    fig.savefig(WORK/name, dpi=175, bbox_inches='tight')
    plt.show()
    plt.close(fig)

render_checks = pd.read_csv(K/'served_prompt_checks.csv')
nll = pd.read_csv(A/'lora_debug_nll.csv')
debug = json.loads((A/'lora_debug_verdict.json').read_text())
errors = pd.read_csv(A/'rollout_error_taxonomy.csv')
assert errors['count'].sum() == 118 and set(errors.total_rollouts) == {280}
assert debug['verdict'] == 'NO-GO'
assert not nll.duplicated(['cohort','span']).any() and not nll.isna().any().any()
assert not render_checks.duplicated(['cohort','rendering','check']).any()


In [ ]:
fig, axs = plt.subplots(1,4,figsize=(13,3.3))
exact = render_checks.query("rendering == 'served_history' and check == 'prompt_token_count'")
mini = debug['criteria']['minieval_resolved_not_worse_than_base_minus_1']
ctrl = nll.query("span == 'overall'").set_index('cohort').relative_change_pct
cards = [
    ('MEASURE', f'{int(audit.sound.sum())} / {len(audit)}', 'Public tasks pass the\npaired-control gate', TEAL),
    ('RENDER', f'{int(exact.matched.sum())} / {int(exact.total.sum())}', 'Prompt lengths match\non the historical stack', BLUE),
    ('CONTROL', f'{-ctrl.rich_resolved:.1f}% | {-ctrl.rich_unresolved:.1f}%', 'Loss falls on resolved |\nunresolved Rich traces', AMBER),
    ('DECIDE', 'NO-GO', f"Adapter {mini['adapter']}/{mini['runs']} vs base {mini['base']}/{mini['runs']}\nLocal task regression guard", RED)]
for ax,(head,value,caption,color) in zip(axs,cards):
    ax.axis('off');ax.axhline(.96,color=color,lw=5,xmin=.04,xmax=.96)
    ax.text(.04,.78,head,color=color,weight='bold',fontsize=11,transform=ax.transAxes)
    ax.text(.04,.47,value,weight='bold',fontsize={'CONTROL':18,'RENDER':19}.get(head,24),transform=ax.transAxes)
    ax.text(.04,.14,caption,fontsize=10.5,linespacing=1.5,transform=ax.transAxes)
fig.suptitle('Lower loss is a measurement. Better repairs are the goal.',x=.04,ha='left',weight='bold',fontsize=17)
fig.subplots_adjust(top=.82,wspace=.22)
save(fig,'01_graphical_abstract.png')


*Figure 1. A chain of local checks: task soundness, historical served-prompt agreement, a resolved/unresolved likelihood control and task-level regression. Sources: `task_soundness.csv`, `served_prompt_checks.csv`, `lora_debug_nll.csv` and `lora_debug_verdict.json` in the attached datasets. Prompt-length equality is weaker than token identity; five live development prompts also matched exact token IDs.*

## 1. Establish which tasks measure a fix

A sound local task should fail with an empty patch and pass with its reference patch, with neither control reporting an error. We ran both controls for all 129 tasks. Dependency-only repairs restored seven Rich environments using their snapshot-pinned Pygments version and two FastAPI environments using declared optional JSON dependencies. The tests, patches and scoring logic remained unchanged.

The final controls produce **119 reference passes** and **2 empty passes**. Soundness also excludes control errors: three Rich empty controls time out despite passing reference controls, and required skipped tests are retained as exclusions. This yields **114 sound tasks and 15 exclusions**. The [official harness](https://www.kaggle.com/competitions/gemma-4-developer-agent/data?select=HARNESS_README.md) supplies the evaluator; our mask is specific to the frozen local environment.


In [ ]:
order=['fastapi/fastapi','Textualize/rich','psf/requests','encode/httpx']
statuses=['sound','passes_without_fix','control_error','gold_does_not_pass_local_grader']
labels=['Sound','Passes empty','Control error','Reference fails']
counts=pd.crosstab(audit.repo,audit.status).reindex(index=order,columns=statuses,fill_value=0)
fig,ax=plt.subplots(figsize=(11,4.3),layout='constrained')
left=np.zeros(len(order))
for status,label,color in zip(statuses,labels,[TEAL,AMBER,RED,GRAY]):
    vals=counts[status].to_numpy();ax.barh(order,vals,left=left,label=label,color=color,height=.60)
    for i,v in enumerate(vals):
        if v>=3:ax.text(left[i]+v/2,i,str(v),ha='center',va='center',color='white' if status=='sound' else '#253b4a',weight='bold')
    left+=vals
for i,repo in enumerate(order):ax.text(left[i]+1,i,f"{counts.loc[repo,'sound']}/{int(left[i])} sound",va='center',fontsize=10)
ax.invert_yaxis();ax.set_xlim(0,85);ax.set_xlabel('Public tasks');ax.set_title('The usable denominator depends on the grader environment')
ax.legend(ncol=4,loc='lower left',bbox_to_anchor=(0,-.30),frameon=False,fontsize=10)
save(fig,'02_soundness.png')
quality=pd.DataFrame({'table':['Task audit','Agent measurements'],'rows':[len(audit),len(runs)],'duplicate keys':[audit.task_id.duplicated().sum(),runs.duplicated(['series','task_id']).sum()],'missing task IDs':[audit.task_id.isna().sum(),runs.task_id.isna().sum()],'missing test exit codes':[audit[['empty_exit_code','gold_exit_code']].isna().sum().sum(),(runs.test_exit_code.eq(-1) | runs.test_exit_code.isna()).sum()]})
display(quality.style.hide(axis='index'))


*Figure 2. FastAPI contributes 62/67 sound tasks, Rich 44/48, Requests 7/13 and HTTPX 1/1. The audit CSV preserves every exclusion reason and both control outcomes. The run table has no blank cells; 62 unavailable verifier exit codes use the recorded sentinel -1. They are not successes or zero-valued timings.*

The repository split assigns FastAPI, Requests and HTTPX to development (**81 tasks, 70 sound**) and Rich to held-out evaluation (**48 tasks, 44 sound**). Each audit identifier and each task–run pair is unique. Repository overlap across splits is zero. Rich is nevertheless only one held-out repository, and the competition's approximately 120 test tasks come from private repositories: this split does not establish performance under that domain shift. [Official data description](https://www.kaggle.com/competitions/gemma-4-developer-agent/data).

For a concrete reading of the table, `fastapi_11194` is sound because empty fails and reference passes. `fastapi_14077` passes without a fix, so it is excluded. `rich_3480` has a timed-out empty control and remains excluded even though its reference passes. These identifiers are join keys; task text and patches must be obtained from the official competition input.

## 2. Compare resolutions on the same sound tasks

The frozen **v4 single agent** has a five-minute cap and 20 tool calls. The **coder + analyzer baseline** gives a coder a separate analysis tool, with an eight-minute cap and 100 tool calls. Both use the required base model with prompt and tool configuration only; neither includes trained weights. Their historical budgets differ, so this comparison measures complete designs rather than the causal effect of adding an analyzer.

We evaluated all 48 held-out tasks for each design, then applied the mask fixed before comparison. The sound-task counts are **14/44 for v4** and **13/44 for coder + analyzer**. Nine successes overlap; five are unique to v4 and four to the baseline. On development, the baseline resolves **15/70** sound tasks: FastAPI 11/62, Requests 3/7, HTTPX 1/1. Errors remain in the denominator.


In [ ]:
s=summary.query("cohort == 'sound_tasks'").set_index('series')
names=['20260930-v4-heldout','20260930-baseline-heldout','20260930-baseline-development']
fig,(ax,bx)=plt.subplots(1,2,figsize=(12,4.4),gridspec_kw={'width_ratios':[1.35,1]},layout='constrained')
for i,(name,color) in enumerate(zip(names,[TEAL,BLUE,GRAY])):
    r=s.loc[name];ax.errorbar(r['rate']*100,i,xerr=[[100*(r['rate']-r.wilson_low)],[100*(r.wilson_high-r['rate'])]],fmt='o',color=color,ms=10,capsize=5,lw=2)
    ax.text(53,i,f'{int(r.resolved)}/{int(r.n)}',va='center',weight='bold')
ax.set_yticks(range(3),['v4 · held-out Rich','Coder + analyzer · Rich','Coder + analyzer · dev'])
ax.invert_yaxis();ax.set_xlim(0,62);ax.set_xlabel('Local resolution rate (%) · Wilson 95% interval');ax.set_title('One fewer resolution, overlapping intervals')
p=protocol['paired_sound_heldout']; vals=[p['both'],len(p['v4_only']),len(p['baseline_only']),p['neither']]
bx.bar(['Both','v4 only','Baseline only','Neither'],vals,color=[TEAL,AMBER,BLUE,GRAY]);bx.bar_label(bx.containers[0]);bx.set_ylim(0,30);bx.set_ylabel('Sound held-out tasks');bx.set_title('The paired outcomes explain the gap')
save(fig,'03_resolution.png')


*Figure 3. Task-level Wilson 95% intervals are 20.0–46.6% for v4, 18.2–44.2% for the held-out baseline and 13.4–32.4% for development. They ignore within-repository correlation and therefore should not be read as cross-repository generalization intervals. Sources: `series_summary.csv` and `protocol.json`.*

The baseline declares seed 42 in YAML, but the installed Agent Development Kit (ADK) conversion omitted that seed from outgoing model requests. The runs are therefore neither deterministic nor matched-seed experiments. Aggregate held-out results informed this report; held-out failure content was not used for prompt selection or training. A better next comparison would match effective seed, budget and server configuration before changing one agent component.

## 3. Price an experiment in seconds

On sound Rich tasks, mean agent-plus-setup time is **228.0 seconds for v4** and **402.9 seconds for the baseline**; mean tool calls are **10.4** and **21.2**. The more expensive design resolves one fewer task in this frozen comparison. Wall time covers setup, agent work and cleanup; local patch grading is excluded to match the competition's stated budget scope. Across all 129 baseline tasks, the mean is **415.0 seconds** and the maximum **584.7 seconds**.

The official allowance is **12 hours including sandbox setup and excluding validation**. Four L4 GPUs form one tensor-parallel (TP4) model in the published configuration. They do not imply four independent task workers. Our measurements come from one A100 with TP1; the L4 slowdown below is an assumption, not a benchmark. [Evaluation budget](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview/evaluation) · [Serving guide](https://www.kaggle.com/competitions/gemma-4-developer-agent/data?select=HARNESS_README.md).


In [ ]:
fig,(ax,bx)=plt.subplots(1,2,figsize=(12,4.7),layout='constrained')
groups=[runs.query('sound == 1 and series == @n').agent_and_setup_seconds.to_numpy() for n in names]
bp=ax.boxplot(groups,vert=False,patch_artist=True,showfliers=False)
ax.set_yticks([1,2,3],['v4 · Rich','Baseline · Rich','Baseline · dev'])
for box,color in zip(bp['boxes'],[TEAL,BLUE,GRAY]):box.set_facecolor(color)
ax.invert_yaxis();ax.set_xlabel('Agent + setup + cleanup (seconds)');ax.set_title('Measured local task cost')
for tasks,color,style in [(60,GRAY,'--'),(120,BLUE,'-'),(129,AMBER,':')]:
    r=runtime.query('assumed_tasks == @tasks');bx.plot(r.assumed_l4_slowdown,r.scenario_hours,label=f'{tasks} assumed tasks',color=color,ls=style,marker='o',lw=2)
bx.axhline(12,color=RED,lw=1.7,label='12-hour limit');bx.set_xlabel('Assumed L4/A100 task-time ratio');bx.set_ylabel('Total scenario (hours)');bx.set_title('Planning scenarios · L4 speed unmeasured');bx.legend(frameon=False,fontsize=9)
save(fig,'04_runtime.png')


*Figure 4. Left: measured distributions on the sound subsets, with medians, interquartile ranges and whiskers within 1.5 interquartile ranges. Right: `(900 + tasks × 414.966 × slowdown × 1.25) / 3600`, using a 15-minute startup allowance and 25% task-time margin. Sources: `local_agent_results.csv` and `runtime_scenarios.csv`.*

For the organizer's approximate **120-task total**, the baseline scenario is **17.54 hours even at an assumed 1× ratio**, and **34.83 hours at 2×**. With the same startup and margin, 12 hours allow about **282 seconds of uninflated task time** per task. A per-task cap alone does not guarantee the total: setup and cleanup consume time, and context overflows or timeouts can spend the budget without returning a useful patch. Hardware calibration and a total-workload dry run are more informative than a larger nominal tool budget.

## 4. Verify that LoRA changes inference

LoRA adds a low-rank update, `ΔW = (alpha / rank) × B × A`, to a base projection. The two official example adapters we inspected have identical weights and zero B tensors. Their zero update is useful as a loading control, but cannot demonstrate nonzero adaptation. The model checkpoint uses quantization-aware training (QAT); the exact serving build matters alongside the adapter configuration. [LoRA paper](https://arxiv.org/abs/2106.09685) · [Official wheelhouse](https://www.kaggle.com/datasets/metric/gemma-4-developer-agent-wheelhouse).

An earlier serving path registered each decoder layer under two names. Adapter activation loaded the canonical name, then reset the same module through an alias without matching weights. The inspected competition wheel avoids that duplicate registration when fast prefill is disabled. The organizer reports that the fix landed in wheelhouse v23. Our result below is tied to its recorded wheel hash, rather than a claim that the present wheelhouse still has the defect. [Alias report and organizer response](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/743508) · [Earlier startup/routing discussion](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/743213).

The strong diagnostic uses **untrained rank-4, alpha-8 weights on 410 projections**. We warm base, zero-adapter and strong-adapter routes, then issue base/zero/strong/base requests with temperature 0 and seed 42 explicitly on the wire. Repeated base and zero control agree exactly. Warmup drift reaches **0.125 nats**, making the frozen ten-times-noise threshold **1.25 nats**. The strong adapter changes the output and removes the base-leading token from its first-token top 20; the resulting decrease is **at least 4.8921 nats**. A nat is one unit of natural-log probability.


In [ ]:
fig,(ax,bx)=plt.subplots(1,2,figsize=(12,4.3),gridspec_kw={'width_ratios':[1,1.3]},layout='constrained')
ax.axis('off')
for y,label,color in [(.8,'Load the adapter',BLUE),(.5,'Check zero + repeat controls',AMBER),(.2,'Require effect above drift',TEAL)]:
    ax.text(.5,y,label,ha='center',va='center',fontsize=12,weight='bold',bbox={'boxstyle':'round,pad=.75','facecolor':color,'alpha':.18,'edgecolor':color},transform=ax.transAxes)
for y in [.69,.39]:ax.annotate('',xy=(.5,y-.1),xytext=(.5,y),xycoords='axes fraction',arrowprops={'arrowstyle':'->','color':'#526574'})
ax.set_title('Loading is the first check')
keys=['base_repeat_drift','zero_adapter_drift','warmup_drift','effect_threshold','strong_effect_lower_bound']
vals=[gate[k] for k in keys]
bars=bx.barh(['Base repeat','Zero adapter','Warmup drift','10× threshold','Strong effect ≥'],vals,color=[GRAY,GRAY,AMBER,BLUE,TEAL],height=.6)
for i,v in enumerate(vals):bx.text(v+.08,i,('≥ ' if i==4 else '')+f'{v:.3f}',va='center',fontsize=10)
bx.invert_yaxis();bx.set_xlim(0,6.1);bx.set_xlabel('First-token log-probability change (nats)');bx.set_title('The synthetic effect clears the threshold')
save(fig,'05_lora_gate.png')
import runpy
probe=runpy.run_path(str(K/'probe_adapter_effect.py'))
replayed=probe['assess'](json.loads((K/'probe_top20.json').read_text()))
assert replayed['pass'] and replayed['missing_token_drop_lower_bound'] > replayed['threshold']
display(Markdown('**CPU replay of the saved effect evidence: PASS.**'))


*Figure 5. The final bar is a conservative lower bound inferred from top-20 membership, not an exactly aligned token delta. The notebook replays that calculation from the attached first-token distributions. This is an inference-effect diagnostic; the synthetic adapter is deliberately untrained and is never packaged as a candidate.*

The successful path uses the competition vLLM **0.19.1** wheel, transformers **5.13.1**, compressed-tensors **0.15.0.1**, torch **2.10.0**, adk-submission **0.2.11** and swegemma **0.2.7**, with 257 frozen compatible packages. The complete historical wheelhouse could not be installed together because of a Starlette dependency conflict; the kit records the compatible lock and exact official wheel hashes. A cluster-specific device-UUID adjustment touched platform plumbing, not model or LoRA code. The private image and TP4 L4 behavior remain unmeasured.

For reproduction, start with the [serving and verification kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit): its CPU replay needs no accelerator; its live probe operates on a separately started, user-owned server. The kit also explains why the adapter map must be nonempty for this ADK version to emit LoRA flags. A trained adapter still needs save/reload checks, long-context memory tests and a matched task comparison after this serving gate.



## 5. Train on the prompt the server actually sees

ADK replays earlier thoughts in `reasoning_content`. In the inspected vLLM 0.19.1 path, `chat_utils` reads `reasoning` instead. The recorded server prompts therefore contain the issue, tools and tool-call history **without past thoughts**. A second detail also matters: vLLM wraps string content as text parts before applying the template. Together, these two changes reproduce **803/803 native** and **133/133 held-out Rich** prompt lengths; the old renderer reproduces **0/803**. Dropping thoughts alone also matches zero; content parts alone match 79 first turns.

This mechanism was already reported by KE WU in [discussion 744354](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/744354). Our contribution is the aggregate ablation, 936-turn length check and reusable reference. The organizer's October 1 reply says the fix should be in the current wheelhouse; September 29 also states that all old submissions cannot be rescored. **The reference documents our historical stack. Recheck the deployed wheels before rendering new training data.**


In [ ]:
from matplotlib.patches import FancyBboxPatch
fig = plt.figure(figsize=(12.5,5.1))
ax = fig.add_axes([.02,.08,.96,.80]);ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off')
boxes=[(.02, 'ADK replay', 'Issue + tools + history\nPast thought in\nreasoning_content', BLUE),
       (.36, 'Historical vLLM parser', 'Reads reasoning only\nDrops reasoning_content\nWraps text as content parts', AMBER),
       (.70, 'Served prefix', 'Issue + tool-call history\nPast thoughts absent\nFresh thought channel opens', TEAL)]
for x,head,body,color in boxes:
    ax.add_patch(FancyBboxPatch((x,.45),.28,.46,boxstyle='round,pad=0.012',facecolor='#f4f7f8',edgecolor=color,lw=2))
    ax.text(x+.14,.83,head,ha='center',weight='bold',fontsize=12,color=color)
    ax.text(x+.14,.63,body,ha='center',va='center',fontsize=11,linespacing=1.7)
for start,end in [(.31,.34),(.65,.68)]:
    ax.annotate('',xy=(end,.68),xytext=(start,.68),arrowprops={'arrowstyle':'->','color':'#253b4a','lw':2})
ax.text(.02,.26,'Measured prompt agreement',weight='bold',fontsize=12)
ax.text(.02,.12,'Old renderer: 0/803 lengths     Served renderer: 803/803 native + 133/133 Rich lengths\nLive token identity: 5/5 development requests',fontsize=12,linespacing=1.7)
fig.suptitle('The inference conditional is defined by the serving path',x=.04,ha='left',weight='bold',fontsize=16)
fig.text(.04,.025,'Historical vLLM 0.19.1 / ADK 0.2.11. Organizer reports a later fix; verify current token IDs.',fontsize=10,color='#5b6d78')
save(fig,'06_served_prompt.png')
display(render_checks.rename(columns={'matched':'matches','total':'checked'}))


*Figure 6. The historical field mismatch removes past reasoning while preserving tool history. Only the five-request live probe checks token identity; the 936-turn result checks prompt lengths. Source: `served_prompt_checks.csv` and `served_render_reference.py` in the serving kit, derived from `research/LORA_DATA_PLAN_20261002.md` §A and `scripts/cluster-debug-iter1/served_render.py`; their hashes are recorded in `source_lineage.json`.*

An empty target thought is another trap. After a tool response, the generation prefix opens `thought`; an empty thought must still close that channel before the action. Rendering a completed assistant message with empty reasoning can skip that opening and violate prefix matching. Reconstructed prompt **and completion** lengths match in **765/803** native turns; exact served token IDs provide a stronger admission rule because text can re-tokenize differently even at the same length. Supervised fine-tuning (SFT) should preserve this per-turn inference conditional and test joint encoding, rather than assuming a whole-trajectory template is equivalent.

The same check changes the external-data decision. Our inspected **150 SWE-smith trajectories** use `bash`, `str_replace_editor` and `submit`, whereas this harness expects `run_command`, `read_file`, `edit_file` and `submit_patch`. Their observations differ, historical thoughts were retained and success labels were not regraded. The native baseline already produces **0 unknown-tool calls in 3,328 tool-call turns**. Those demonstrations are a poor fit for this SFT mix without new execution and grading; this is a sample-specific finding. The kit's `external_trajectory_assessment.json` provides the aggregate evidence from the data plan's section B.

## 6. Use an unresolved-trace control before trusting a loss reduction

Negative log-likelihood (NLL) measures how much probability the model assigns to the recorded target tokens. It is useful for debugging, but a model trained on its own successful traces can become more confident in its existing style without becoming a better repair agent. We test this with both resolved and unresolved Rich traces and a resolved set from the different coder/analyzer design.

The debug adapter uses rank 16, alpha 32, learning rate 1e-4 and seed 42. Training admits **412 turns from 23 development tasks**, caps sequences at 8,192 tokens, excludes Rich and three development-validation tasks, and runs 30 optimizer steps with accumulation eight. The supervised data are mostly thoughts: 97,829 thought tokens and 19,464 action tokens. The final adapter reduces Rich action NLL by **40.7%** and thought NLL by **25.5%**.


In [ ]:
fig,(ax,bx)=plt.subplots(1,2,figsize=(12,4.8),layout='constrained',gridspec_kw={'width_ratios':[.85,1.3]})
span=nll.query("cohort == 'rich_resolved' and span != 'overall'").set_index('span')
ax.barh(['Action','Thought'],[-span.loc['action','relative_change_pct'],-span.loc['thought','relative_change_pct']],color=[BLUE,TEAL],height=.55)
ax.invert_yaxis();ax.set_title('The attractive signal',fontsize=13)
controls=nll.query("span == 'overall' and cohort != 'development_validation'").set_index('cohort')
cohorts=['rich_resolved','rich_unresolved','coder_analyzer_resolved']
values=[-controls.loc[x,'relative_change_pct'] for x in cohorts]
bx.barh(['Rich: resolved','Rich: unresolved','Other agent: resolved'],values,color=[TEAL,AMBER,GRAY],height=.55)
bx.invert_yaxis();bx.set_title('The sharpening control',fontsize=13)
for panel in [ax,bx]:
    panel.set_xlim(0,48);panel.set_xlabel('NLL reduction from base (%)')
    panel.grid(axis='x',alpha=.15);panel.set_axisbelow(True)
    for bar in panel.patches:
        panel.text(bar.get_width()+.6,bar.get_y()+bar.get_height()/2,f'{bar.get_width():.1f}%',va='center',fontsize=11)
fig.suptitle('Loss improves on unsuccessful traces almost as much as successful ones',fontsize=15,weight='bold')
save(fig,'07_sharpening_control.png')
criteria=debug['criteria']
display(pd.DataFrame([
 {'Check':'Rich overall NLL (nats/token)','Base or mid':criteria['final_not_worse_than_mid_on_rich']['mid'],'Final adapter':criteria['final_not_worse_than_mid_on_rich']['final'],'Interpretation':'Final worse than mid: FAIL'},
 {'Check':'Resolved / 32 local runs','Base or mid':4,'Final adapter':2,'Interpretation':'Allowed loss at most one: FAIL'},
 {'Check':'Mean agent wall time ratio','Base or mid':1,'Final adapter':criteria['agent_wall_time_within_10pct']['ratio'],'Interpretation':'Within +10%: PASS'},
 {'Check':'Completion tokens per turn ratio','Base or mid':1,'Final adapter':criteria['completion_tokens_per_turn_within_10pct']['ratio'],'Interpretation':'Within +10%: PASS'}]))


*Figure 7. Left: span-specific loss reductions on resolved Rich turns. Right: overall reductions of **27.1%** on resolved Rich, **25.5%** on unresolved Rich and **26.4%** on the other agent's resolved traces. The 1.6-percentage-point resolved/unresolved gap is small and has no reported uncertainty estimate; the reductions mainly support broad sharpening, not a demonstrated quality gain. Sources: `lora_debug_nll.csv` and `lora_debug_verdict.json`, derived from `experiments/runs/20261002-fable-debug-iter1/verdict.json`.*

The registered gate returns **NO-GO** for two reasons. Final Rich overall NLL rises to **0.07342** from the mid checkpoint's **0.06779**, and the matched mini-evaluation resolves **2/32** runs with the adapter versus **4/32** with base. Wall time rises **1.7%** and completion tokens per turn **3.7%**, both within the permitted 10% overhead. Tool validity passes. Eight of ten checks pass, but the two failures are sufficient to stop promotion.

The mini-evaluation uses **16 Rich tasks × seeds 42 and 7**, a two-minute task cap and eight concurrent agents on one A100 tensor-parallelism-1 LoRA server. It is a regression guard with low power for small improvements. NLL evaluation uses only 50 resolved Rich turns, 30 unresolved Rich turns, 24 other-agent turns and 30 development-validation turns, each capped at 8,192 tokens. Rich is the sole held-out repository and is now repeatedly inspected diagnostic evidence; future final claims require an independent evaluation. The debug adapter is not a submission candidate. Protocol and limitations come from `research/LORA_DEBUG_ITER1_20261002.md`; final measurements come from the completed verdict JSON, both identified by hashes in the dataset lineage.

## 7. Separate infrastructure failures from model behavior

Among **280 historical rollouts**, **118** record errors: **81** five-minute session timeouts, **32** snapshot extraction failures, **3** exhausted 60-turn budgets and **2** context overflows. Every error remains unresolved. Of the 32 extraction failures, 21 happen before model traffic and 11 during grading. The 162 other outcomes have no recorded error; that does not make all of them successful. Sources: `rollout_error_taxonomy.csv`, derived from `research/ROLLOUT_ERRORS_20261002.md`, and the hashed classification JSON.

A separate historical Network File System (NFS) reproduction at concurrency four hits the **60-second extraction timeout in 4/4 cases**, with exit 124. With the extraction limit alone raised to 600 seconds, four reruns pass in roughly 56 seconds, likely benefiting from warm caches. A subsequent live extraction reaches **128.467 seconds**, exceeding the original allowance. This supports a separate extraction budget and duration/exit-code logging. It does not establish the cause of every older failure, whose stderr and exit status were absent. `extraction_timing_note.json` contains only this completed infrastructure diagnostic, derived from the extraction section of `research/LORA_ITER2_HANDOFF_20261002.md`.

Session, tool and grading limits should remain explicit when the extraction allowance changes. Faster setup increases usable compute; it is not evidence that the agent has learned to repair more tasks.


## 8. Place the evidence beside existing work

The [official starter](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent) teaches the complete inference and packaging path. [Roman Rozen's baseline](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1) develops code localization and a coder/analyzer design; it informed our baseline. [Black Cat](https://www.kaggle.com/code/lucifer19/black-cat-swe-agent-pack-instinct) explores agent tools and skills. [Oleksii Zhukov's walkthrough](https://www.kaggle.com/code/zhukovoleksiy/gemma-4-walkthrough-first-submission) consolidates serving and harness failure modes.

[Busya PRIME's audit](https://www.kaggle.com/code/busyaprime/119-of-129-sound-the-gemma-4-grader-rebuilt) already makes the case for paired controls and reports 119 sound tasks under its environment and treatment of skipped nodes. Our **114** is a separate frozen mask from the released evaluator with dependency-only repairs and conservative control-error exclusions. The contribution here is the joined evidence: that mask, 177 task-run outcomes, a replayable nonzero-adapter gate, served-rendering checks and a trained-adapter NO-GO with likelihood controls. Neither audit count is a statement about defects in the hidden benchmark. The issue-to-repository-patch setup follows the benchmark tradition introduced by [SWE-bench](https://arxiv.org/abs/2310.06770); our repository split is a separate local design choice.

The [Public LB snapshot](https://www.kaggle.com/competitions/gemma-4-developer-agent/leaderboard) checked October 3 shows a leader at **0.24** and three teams at **0.17**. The discussions and public notebooks reviewed for this release do not establish the method behind those scores. The organizer-reported reasoning fix changes the inference path, but attributing the score jump to that fix or to LoRA would require evidence that is not public in the reviewed sources. [Reasoning-persistence update](https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/744354).

## 9. Build the submitted archive: the public 0.12 reference agent

The archive below reproduces, file for file, the coder/analyzer bundle published by [Roman Rozen](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1) (Apache 2.0), which scores **0.12** on the Public LB from his notebook. It is the same design family as the coder/analyzer baseline measured above (temperature 0.2, top-k 40, 8,192 output tokens, a 4,096-token thinking budget, a read-only analyzer called as a tool, harness-default budgets). We ship it unchanged as the reference agent for this notebook's measurements: the contribution here is the measurement layer, and every candidate change above should be judged against this baseline.

The cell checks the bundle text against a content hash, verifies the declared model, sampling, tool sets and include safety, runs the released compiler when installed and writes a deterministic `submission.zip` plus `archive_manifest.json`. Our earlier evidence-first prompt variant scored 0.08 (submission 56700442) and is retired here. This notebook performs CPU analysis and packaging only; it does not launch model inference.


In [ ]:
# Exact text of the public 0.12 bundle by Roman Rozen (Apache 2.0), output of
# https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1
FILES = {
 "agent.yaml": "name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that fixes repository issues with minimal, verified patches.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\n",
 "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\ntop_k: 40\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: false\n",
 "prompts/analyzer.md": "You are `code_analyzer`, a read-only code navigation specialist. You never modify files.\nGiven an issue, find exactly where it must be fixed.\n\n## Tools\n- `run_command` for READ-ONLY commands only: `grep -rn`, `ls`, `sed -n`, `git log -p -S`\n- `search_similar_code` for concepts the issue describes without naming code\n- `get_code_neighbors` to walk callers and callees\n- `get_code_subgraph` to see how a few candidate symbols connect\n- `read_file` with tight line ranges to confirm\n\n## Method\n1. Extract identifiers from the issue: function/class names, error messages, file paths, options.\n2. Search for each one, then follow the call chain until you reach the line where behaviour diverges from what the issue expects.\n3. Confirm by reading the actual code. Never guess line numbers.\n\n## Answer format (at most 250 words, nothing else)\nLOCATION: <path>:<start>-<end> (<function or class>)\nROOT CAUSE: <one or two sentences>\nFIX PLAN: <concrete change>\nRELATED: <other call sites or files needing the same change, or \"none\">\nTESTS: <existing test files that exercise this code>\nCONFIDENCE: high | medium | low\n",
 "prompts/system.md": "You are an autonomous senior Python engineer working inside a sandboxed checkout of a real open-source repository at /workspace.\nGoal: resolve the issue in the user message with the smallest correct patch, then call `submit_patch`.\n\n## Hard rules\n- Never edit, add or delete tests, `conftest.py`, `pytest.ini`, CI or packaging files. Hidden tests are applied after you finish.\n- Keep public APIs backward compatible unless the issue explicitly asks for a change.\n- Scratch files go to /tmp only. Anything left in /workspace becomes part of your patch.\n- The environment is pre-built: do not try to install packages.\n- Always finish by calling `submit_patch`. A careful best-effort fix beats no patch.\n\n## Workflow\n1. **Understand**: state the expected vs. actual behaviour to yourself in one or two sentences.\n2. **Localize**:\n   - Call the `code_analyzer` tool with the full issue text first. It returns LOCATION / ROOT CAUSE / FIX PLAN. Verify its claim by reading those exact lines before editing.\n   - Extract every identifier, error message and file name from the issue and search for them: `grep -rn \"<identifier>\" --include=*.py . | head -30`.\n   - Read only the lines you need (`read_file` with a line range or `sed -n 'START,ENDp' FILE`).\n3. **Reproduce**: write a minimal script to /tmp/repro.py that shows the bug and run it with `python /tmp/repro.py`.\n4. **Fix**: edit source files with `edit_file`. Copy `old_string` verbatim from the file, *including leading indentation*, and strip any line-number prefixes. Keep `old_string` short but unique. One logical change per edit. Fix the root cause, not the symptom, and also handle the edge cases the issue mentions.\n5. **Verify**: run `python -m py_compile <file>` after every edit, rerun /tmp/repro.py, then run the closest existing tests: `python -m pytest <tests/path> -x -q` (narrow with `-k`).\n6. **Submit**: run `git status` and `git diff`, make sure only intended source changes remain, then call `submit_patch`.\n\n## Budget discipline\n- Call `get_status` every ~8 tool calls. When less than 25% of turns or time remain, stop exploring and go straight to Fix → Verify → Submit.\n- Keep outputs short: pipe through `head`, use `grep -n`, `pytest -q`. Never print whole large files.\n- If an edit fails twice, re-read the exact lines and retry with a smaller unique snippet.\n\n## Quality bar\n- Match the surrounding code style, type hints and naming.\n- Prefer a small, targeted change over a refactor. Touch other files only when the fix requires it.\n",
 "sub_agents/code_analyzer.yaml": "name: code_analyzer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Read-only code navigator. Given an issue, returns the exact location, root cause and a fix plan.\ninstruction: !include ../prompts/analyzer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n"
}
BUNDLE_CONTENT_SHA256 = "1ef16e9ccf23d2e4facb61c8ec24d67dae86a7a20c2fecb1544f857f2e1ca3d3"
assert hashlib.sha256(json.dumps(FILES, sort_keys=True).encode()).hexdigest() == BUNDLE_CONTENT_SHA256

BUNDLE = WORK / "submission_bundle"
for name, text in FILES.items():
    path = BUNDLE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def load_included(path):
    class Loader(yaml.SafeLoader):
        pass
    def include(loader, node):
        target = (path.parent / loader.construct_scalar(node)).resolve()
        assert target.is_relative_to(BUNDLE.resolve()), target
        if target.suffix in (".yaml", ".yml"):
            return load_included(target)
        return target.read_text(encoding="utf-8")
    Loader.add_constructor("!include", include)
    return yaml.load(path.read_text(encoding="utf-8"), Loader=Loader)

root = load_included(BUNDLE / "agent.yaml")
analyzer = load_included(BUNDLE / "sub_agents/code_analyzer.yaml")
MODEL = "gemma-4-31b-it-qat-w4a16-ct"
assert root["model"] == analyzer["model"] == MODEL
cfg = root["generate_content_config"]
assert (cfg["temperature"], cfg["top_k"], cfg["max_output_tokens"]) == (0.2, 40, 8192)
assert {x for x in root["tools"] if isinstance(x, str)} == {"run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch"}
assert root["tools"][-1]["agent_tool"]["config_path"] == "sub_agents/code_analyzer.yaml"
assert "edit_file" not in analyzer["tools"] and "write_file" not in analyzer["tools"]

compile_status = "released compiler unavailable; strict static validation passed"
try:
    from adk_submission import compile_submission, validate_directory
    from swegemma.config import build_submission_limits
    from swegemma.models import setup_gemma_model_registry, validate_single_declared_model
    from swegemma.tools import create_tools
except ImportError:
    pass
else:
    limits, generation_constraints = build_submission_limits()
    validate_directory(BUNDLE, limits)
    assert validate_single_declared_model(BUNDLE) == MODEL
    agent = compile_submission(
        submission_dir=BUNDLE,
        tool_registry=create_tools(object()),
        model_registry=setup_gemma_model_registry(api_base="http://127.0.0.1:8000/v1", api_key="EMPTY"),
        limits=limits, generation_constraints=generation_constraints)
    assert agent.name == root["name"]
    compile_status = "released ADK compiler PASS"

ARCHIVE = WORK / "submission.zip"
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    for name in sorted(FILES):
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.create_system = 3
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        zf.writestr(info, FILES[name].encode("utf-8"), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)
with zipfile.ZipFile(ARCHIVE) as zf:
    assert zf.namelist() == sorted(FILES)
    assert all(zf.read(name) == FILES[name].encode("utf-8") for name in FILES)
import shutil; shutil.rmtree(BUNDLE)
manifest = {"archive": ARCHIVE.name, "sha256": hashlib.sha256(ARCHIVE.read_bytes()).hexdigest(),
            "bytes": ARCHIVE.stat().st_size, "files": sorted(FILES), "bundle_content_sha256": BUNDLE_CONTENT_SHA256,
            "source": "romanrozen/gemma-eda-baseline-for-a-start-lb-top-1 output bundle (Public LB 0.12 there)",
            "validation": compile_status}
(WORK / "archive_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


## What to prioritize next

**First, freeze the measurement path.** Reuse a documented soundness mask, keep repositories disjoint and match task budgets, actual request seeds and exact serving wheels. Capture served token IDs when available. After a reasoning-persistence update, recheck both the SFT rendering and context budget.

**Second, use controls that can reject an attractive story.** Compare resolved and unresolved traces, include a different agent design and track both intermediate and final checkpoints. Rank candidates by matched task resolution, regressions and complete runtime. In this debug run, lower loss is insufficient to promote the adapter.

**Third, recover useful compute before scaling training.** Report extraction failures separately, bound repeated reads and measure setup, generation and grading costs. External demonstrations need compatible tools, observation formats, licensing and regraded labels. Complementary agents justify an ensemble only after its extra calls fit a measured workload budget. [External-data rules](https://www.kaggle.com/competitions/gemma-4-developer-agent/rules).

The reusable outcome is a tested evidence chain: audit tasks, verify what the model sees, distinguish confidence from repair quality and retain failed experiments. The next quality claim should come from fresh independent tasks, matched evaluation and measured target-hardware runtime.

**Data and reuse.** [Task soundness and local results](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-task-soundness-and-local-results) · [Serving and LoRA verification kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit). Both include file and column descriptions, source hashes, quick starts and limitations. Original measurements/documentation are CC BY 4.0; original kit scripts additionally use Apache 2.0. Official task content, weights, wheels, trajectories, training data, the private extra-task pool and current iteration-2 artifacts are excluded.
